<a href="https://colab.research.google.com/github/JoeVPhD/pioreactor-od-calculator/blob/main/PioReactior_OD-Calculator-and-Corrector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OD Reading vs. Hours Since Experiment

This notebook lets you load a CSV file and plot `od_reading` vs `hours_since_experiment_created`, with one trace per pioreactor unit.

It also includes optional normalization cells so you can compare trajectories across units after baseline scaling.

In [1]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from scipy.signal import medfilt
from scipy.stats import linregress
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG') is not None
print('Running in Colab:', IN_COLAB)

Running in Colab: True


## 1. Select File

In [ ]:
import os, sys

if IN_COLAB:
    from google.colab import files
    from pathlib import Path
    print("Running in Google Colab — please upload your CSV file:")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    CSVPATH = Path(fname)
else:
    import tkinter as tk
    from tkinter import filedialog
    from pathlib import Path
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    CSVPATH = filedialog.askopenfilename(
        title='Select CSV file',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )
    root.destroy()
    if not CSVPATH:
        raise FileNotFoundError('No file was selected.')
    CSVPATH = Path(CSVPATH).expanduser().resolve()

print(f'Selected file: {CSVPATH}')

Running in Google Colab — please upload your CSV file:


## 2. Load Data

In [ ]:
# ── File selection ──────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files
    print('Upload your CSV file:')
    uploaded = files.upload()
    CSVPATH = Path(list(uploaded.keys())[0])
else:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
    CSVPATH = Path(filedialog.askopenfilename(
        title='Select CSV file',
        filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
    )).expanduser().resolve()
    root.destroy()

# ── Fast load: skip date parsing, load only needed columns, explicit dtypes ─
COLS   = ['pioreactor_unit', 'od_reading', 'channel', 'hours_since_experiment_created']
DTYPES = {
    'pioreactor_unit':                'category',
    'od_reading':                     'float32',
    'channel':                        'int8',
    'hours_since_experiment_created': 'float32',
}

df = pd.read_csv(CSVPATH, usecols=COLS, dtype=DTYPES)
df = df.rename(columns={
    'hours_since_experiment_created': 'hourssince',
    'pioreactor_unit':                'pioreactor',
})

OUTPUTDIR = CSVPATH.parent
print(f'Loaded: {len(df):,} rows  |  Units: {sorted(df["pioreactor"].unique())}')
print(f'Output dir: {OUTPUTDIR}')

## 3. Optional Downsample for Fast Plotting

The dataset can be large. If plotting is slow, uncomment the downsample cell below
to keep every Nth point per unit (a rolling median is applied first to reduce noise).

In [ ]:
#CHANNEL = None  # Set to e.g. 2 to filter to a single channel

#dfplot = df[df['channel'] == CHANNEL].copy() if CHANNEL is not None else df.copy()
#print(f'Rows for plotting: {len(dfplot):,}')

#units   = sorted(dfplot['pioreactor'].unique())
#palette = sns.color_palette('tab10', len(units))
#ncols   = 3
#nrows   = -(-len(units) // ncols)

## 4. Create Normalized OD Values

This section creates normalized versions of `od_reading` for each pioreactor.

Included normalizations:
- `od_norm_t0`: divides by the first OD value for each pioreactor.
- `od_delta_t0`: subtracts the first OD value for each pioreactor.
- `od_minmax`: rescales each pioreactor trace between 0 and 1.

In [ ]:
g        = dfplot.groupby('pioreactor', observed=True)['od_reading']
first_od = g.transform('first')
min_od   = g.transform('min')
max_od   = g.transform('max')
range_od = (max_od - min_od).replace(0, pd.NA)

dfplot = dfplot.copy()
dfplot['od_norm_t0']  = dfplot['od_reading'] / first_od
dfplot['od_delta_t0'] = dfplot['od_reading'] - first_od
dfplot['od_minmax']   = (dfplot['od_reading'] - min_od) / range_od

dfplot[['pioreactor','hourssince','od_reading','od_norm_t0','od_delta_t0','od_minmax']].head()

## 5.  Pre-compute Per-Unit Arrays

---



In [ ]:
# Sorts and computes rolling median ONCE per unit — all plot cells below reuse this cache
SMOOTH_WIN = 60

unit_data = {}
for unit in units:
    grp = dfplot[dfplot['pioreactor'] == unit].sort_values('hourssince')
    t   = grp['hourssince'].values
    od  = grp['od_reading'].values
    sm  = grp['od_reading'].rolling(SMOOTH_WIN, center=True, min_periods=1).median().values
    unit_data[unit] = {
        't':        t,
        'od':       od,
        'smoothed': sm,
        'norm_t0':  grp['od_norm_t0'].values,
        'delta_t0': grp['od_delta_t0'].values,
        'minmax':   grp['od_minmax'].values,
    }

print('Pre-computation complete. Cached units:', list(unit_data.keys()))

## 6. Plot Normalised OD (t₀)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for unit, color in zip(units, palette):
    d = unit_data[unit]
    ax.plot(d['t'], d['norm_t0'], label=unit, color=color, alpha=0.8, linewidth=0.8)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('OD / OD(t=0)')
ax.set_title('Normalised OD (relative to first reading)')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_norm_t0.png', dpi=150)
plt.show()

## 7. Plot: Raw OD — All Units Overlays

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for unit, color in zip(units, palette):
    d = unit_data[unit]
    ax.plot(d['t'], d['od'], label=unit, color=color, alpha=0.7, linewidth=0.8)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title('od_reading vs. Time — All Pioreactor Units')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_all_units.png', dpi=150)
plt.show()

## 8. Plot Raw OD — Individual Unit Subplots

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for unit, color in zip(units, palette):
    d = unit_data[unit]
    ax.plot(d['t'], d['od'],       color=color, alpha=0.2, linewidth=0.5)
    ax.plot(d['t'], d['smoothed'], color=color, linewidth=1.5, label=unit)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title(f'od_reading vs. Time — Smoothed (rolling median, window={SMOOTH_WIN})')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_smoothed.png', dpi=150)
plt.show()

## 9 Plot Smoothed OD — Rolling Median Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for unit, color in zip(units, palette):
    d = unit_data[unit]
    ax.plot(d['t'], d['od'],       color=color, alpha=0.2, linewidth=0.5)
    ax.plot(d['t'], d['smoothed'], color=color, linewidth=1.5, label=unit)
ax.set_xlabel('Hours since experiment created')
ax.set_ylabel('od_reading')
ax.set_title(f'od_reading vs. Time — Smoothed (rolling median, window={SMOOTH_WIN})')
ax.legend(title='Unit', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_vs_hours_smoothed.png', dpi=150)
plt.show()

## 10. Find Doubling Time — Exponential Fit

Finds the best local exponential growth window per unit using a sliding window on log(OD),
then reports growth rate *r* (h⁻¹), doubling time *Td* = ln(2)/r, and R².

In [ ]:
WIN_H     = 3.0   # sliding window width in hours
MIN_OD    = 0.05  # ignore readings below this (noise floor)
MEDFILT_K = 51    # robust smoothing kernel (must be odd)

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows), sharey=False)
axes = axes.flatten()
palette_exp = sns.color_palette('tab10', len(units))

for i, (unit, color) in enumerate(zip(units, palette_exp)):
    d  = unit_data[unit]
    t  = d['t']
    od = d['od']

    mask_od = od > MIN_OD
    t_f, od_f = t[mask_od], od[mask_od]
    if len(t_f) < 20:
        continue

    k    = min(MEDFILT_K, len(od_f) if len(od_f) % 2 == 1 else len(od_f) - 1)
    od_s = medfilt(od_f, kernel_size=k)
    with np.errstate(divide='ignore', invalid='ignore'):
        log_od = np.where(od_s > 0, np.log(od_s), np.nan)

    # searchsorted replaces boolean mask — much faster on large arrays
    best_r2, best_r, best_td, best_slice = -np.inf, np.nan, np.nan, None
    best_intercept = np.nan
    for j in range(len(t_f)):
        lo = j
        hi = int(np.searchsorted(t_f, t_f[j] + WIN_H, side='right'))
        if hi - lo < 5:
            continue
        sl = slice(lo, hi)
        y  = log_od[sl]
        if np.any(np.isnan(y)):
            continue
        slope, intercept, r, _, _ = linregress(t_f[sl], y)
        if r**2 > best_r2 and slope > 0:
            best_r2, best_r, best_td = r**2, slope, np.log(2) / slope
            best_slice, best_intercept = sl, intercept

    ax = axes[i]
    ax.plot(t_f, od_f, color=color, alpha=0.2, linewidth=0.5)
    ax.plot(t_f, od_s, color=color, linewidth=1.5)
    if best_slice is not None:
        t_win = t_f[best_slice]
        ax.fill_betweenx([0, od_f.max()*1.1], t_win[0], t_win[-1], color='gray', alpha=0.15)
        ax.plot(t_win, np.exp(best_intercept + best_r*t_win), 'k--', linewidth=1.5)
        ax.text(0.02, 0.97,
                f'r={best_r:.3f} h⁻¹\nTd={best_td:.2f} h\nR²={best_r2:.3f}',
                transform=ax.transAxes, va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
    ax.set_title(unit, fontweight='bold')
    ax.set_xlabel('Hours since experiment created')
    ax.set_ylabel('od_reading')
    ax.grid(True, linestyle='--', alpha=0.5)

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
fig.suptitle('od_reading with robust smoothing and best local exponential fit',
             fontweight='bold', fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUTDIR / 'od_reading_robust_expfit_subplots.png', dpi=150)
plt.show()